# Render 10 Video Sekaligus (5 Sejarah + 5 Stickman) di Google Colab (T4 GPU)

Notebook render otomatis untuk **10 video sekaligus** secara berurutan (1920x1080 @ 30fps, hard-cut statis tanpa transisi, sinkron kata kunci visual + *WAV vocal onset*):

### 5 Video Sejarah
1. **Sejarah Persia** (`SejarahPersia`) — ~12,94 menit | 208 scenes
2. **Sejarah Yunani** (`SejarahYunani`) — ~13,40 menit | 161 scenes
3. **Sejarah Majapahit** (`SejarahMajapahit`) — ~10,52 menit | 148 scenes
4. **Sejarah Roma** (`SejarahRoma`) — ~13,64 menit | 269 scenes
5. **Sejarah China** (`SejarahChina`) — ~10,68 menit | 251 scenes

### 5 Video Stickman
6. **Orang Indonesia Pertama** (`OrangIndoPertama`) — ~5,11 menit | 95 scenes
7. **Piramida Mesir Kuno** (`PiramidaStickman`) — ~8,33 menit | 151 scenes
8. **Misteri Rekayasa Taj Mahal** (`TajMahalStickman`) — ~6,41 menit | 91 scenes
9. **Evolusi Warna Kulit Manusia** (`KulitStickman`) — ~6,20 menit | 80 scenes
10. **Eksperimen 11 Hari Tak Tidur** (`TakTidurStickman`) — ~6,30 menit | 110 scenes

---
### Petunjuk Penggunaan:
1. Pastikan runtime Colab menggunakan **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Jalankan **Sel 1 (Mount Google Drive)** agar seluruh hasil render tersimpan permanen di Google Drive (`remotion-out/`).
3. Jalankan **Sel 2 (Setup Environment)** untuk instalasi dependensi Chromium, Node.js 20, dan clone/update repo.
4. Jalankan **Sel 3 (Render Otomatis 10 Video)** — mendukung *auto-resume* (`SKIP_EXISTING = True`) sehingga jika sesi Colab terputus di tengah jalan, jalankan ulang Sel 3 untuk melanjutkan video yang belum selesai.
5. Jalankan **Sel 4 (Verifikasi & Download)** untuk mengecek ukuran seluruh 10 file video.


In [ ]:
# Sel 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
out_dir = '/content/drive/MyDrive/remotion-out'
os.makedirs(out_dir, exist_ok=True)
print('Google Drive terhubung! Output 10 video akan disimpan di:', out_dir)


In [ ]:
%%shell
# Sel 2: Setup Environment, Dependensi Chromium & Clone Repo
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak terdeteksi! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install Dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai 100%! Siap merender 10 video (5 Sejarah + 5 Stickman)."


In [ ]:
# Sel 3: Render Otomatis 10 Video (5 Sejarah + 5 Stickman) Berurutan
import importlib, os, subprocess, sys, time

# Set True agar video yang sudah selesai dirender di Drive otomatis dilewati (aman untuk resume)
# Set False jika ingin merender ulang semuanya dari awal
SKIP_EXISTING = True

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
os.makedirs(out_dir, exist_ok=True)

if '/content/vidio/scripts' not in sys.path:
    sys.path.insert(0, '/content/vidio/scripts')

import colab_render
importlib.reload(colab_render)

start_all = time.time()

tasks = [
    # 5 Video Sejarah
    ('SejarahPersia',    'sejarah-persia-1080p.mp4',      '1. Sejarah Persia (~12,94 Menit)'),
    ('SejarahYunani',    'sejarah-yunani-1080p.mp4',      '2. Sejarah Yunani (~13,40 Menit)'),
    ('SejarahMajapahit', 'sejarah-majapahit-1080p.mp4',   '3. Sejarah Majapahit (~10,52 Menit)'),
    ('SejarahRoma',      'sejarah-roma-1080p.mp4',        '4. Sejarah Roma (~13,64 Menit)'),
    ('SejarahChina',     'sejarah-china-1080p.mp4',       '5. Sejarah China (~10,68 Menit)'),
    # 5 Video Stickman
    ('OrangIndoPertama', 'orang-indo-pertama-1080p.mp4',  '6. Orang Indonesia Pertama (~5,11 Menit)'),
    ('PiramidaStickman', 'piramida-mesir-1080p.mp4',      '7. Piramida Mesir (~8,33 Menit)'),
    ('TajMahalStickman', 'tajmahal-1080p.mp4',            '8. Misteri Taj Mahal (~6,41 Menit)'),
    ('KulitStickman',    'kulit-1080p.mp4',               '9. Evolusi Warna Kulit (~6,20 Menit)'),
    ('TakTidurStickman', 'tak-tidur-1080p.mp4',           '10. Eksperimen Tak Tidur 1964 (~6,30 Menit)'),
]

total_tasks = len(tasks)
for idx, (comp_id, out_name, label) in enumerate(tasks, start=1):
    target_path = os.path.join(out_dir, out_name)
    if SKIP_EXISTING and os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
        size_mb = os.path.getsize(target_path) / (1024 * 1024)
        print(f'>>> [{idx}/{total_tasks}] SKIP (sudah ada): {out_name} ({size_mb:.2f} MB)')
        continue

    print(f'\n=====================================================================')
    print(f'=== [{idx}/{total_tasks}] MEMULAI RENDER: {label} ===')
    print(f'=== Komposisi: {comp_id} -> {out_name} ===')
    print(f'=====================================================================')
    t0 = time.time()
    colab_render.run_render(
        composition=comp_id,
        out_path=target_path,
        cwd='/content/vidio',
    )
    t_dur = (time.time() - t0) / 60.0
    print(f'>>> [{idx}/{total_tasks}] Selesai dalam {t_dur:.2f} menit! File: {target_path}')

total_min = (time.time() - start_all) / 60.0
print(f'\n[SELESAI 100%] Seluruh {total_tasks} video selesai diproses dalam total {total_min:.2f} menit!')
print('Semua hasil video tersimpan di:', out_dir)


In [ ]:
# Sel 4: Verifikasi Ukuran 10 File Video & Opsi Download ke Lokal
import os
from google.colab import files

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
video_files = [
    os.path.join(out_dir, 'sejarah-persia-1080p.mp4'),
    os.path.join(out_dir, 'sejarah-yunani-1080p.mp4'),
    os.path.join(out_dir, 'sejarah-majapahit-1080p.mp4'),
    os.path.join(out_dir, 'sejarah-roma-1080p.mp4'),
    os.path.join(out_dir, 'sejarah-china-1080p.mp4'),
    os.path.join(out_dir, 'orang-indo-pertama-1080p.mp4'),
    os.path.join(out_dir, 'piramida-mesir-1080p.mp4'),
    os.path.join(out_dir, 'tajmahal-1080p.mp4'),
    os.path.join(out_dir, 'kulit-1080p.mp4'),
    os.path.join(out_dir, 'tak-tidur-1080p.mp4'),
]

print('Daftar 10 file video hasil render di Drive:')
done = 0
for v in video_files:
    if os.path.exists(v):
        size_mb = os.path.getsize(v) / (1024 * 1024)
        print(f'  [OK] {os.path.basename(v)}: {size_mb:.2f} MB')
        done += 1
    else:
        print(f'  [--] {os.path.basename(v)}: Belum selesai / tidak ditemukan')
print(f'Total selesai: {done}/{len(video_files)} video')

# Uncomment baris di bawah ini jika ingin men-download otomatis ke laptop:
# for v in video_files:
#     if os.path.exists(v):
#         files.download(v)
